# Statistical Analysis of Book Prices and Ratings
**Subject:** Statistical Inference (SI) Lab – Mini Project  
**Data source:** [Books to Scrape](https://books.toscrape.com) (a practice site built for scraping)

This project combines the techniques from the eight practicals:

| Section | Practical | Technique |
|---|---|---|
| 1 | Lab 1 & 12 | Web scraping with `requests` + `BeautifulSoup` |
| 2 | Lab 2A | Cleaning and descriptive statistics |
| 3 | Lab 2B | Visualisation (histogram, box, bar, pie, scatter) |
| 4 | Lab 3 | Probability and distributions |
| 5 | Lab 4 | Sampling distribution and Central Limit Theorem |
| 6 | Lab 5 | Confidence intervals, Z-test, one-sample t-test |
| 7 | Lab 6 | Chi-square, two-sample t-test, ANOVA, correlation |
| 8 | Lab 7 & 8A | Simple and multiple regression, Ridge, Lasso, Tree, Forest |
| 9 | Lab 8B | Classification: predict high-rated books |
| 10 | Lab 7 | Export data **and trained models (.pkl)** for the Streamlit app |

**Research questions**
1. Is a book's price related to its rating?
2. Do prices differ across categories?
3. Can we predict price, or whether a book is highly rated?

## 1. Data Collection (Web Scraping)

In [ ]:
import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from scipy import stats
from scipy.stats import chi2_contingency, norm, binom
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [ ]:
HEADERS = {"User-Agent": "StudentLabBot/1.0 (educational project)"}
RATING = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}
INDEX = "https://books.toscrape.com/catalogue/category/books_1/index.html"

def scrape_books():
    r = requests.get(INDEX, headers=HEADERS, timeout=20); r.raise_for_status()
    r.encoding = "utf-8"
    soup = BeautifulSoup(r.text, "html.parser")
    cats = [(a.text.strip(), urljoin(INDEX, a["href"]))
            for a in soup.select("ul.nav-list ul li a")]
    rows = []
    for cat, url in cats:
        while url:
            r = requests.get(url, headers=HEADERS, timeout=20); r.encoding = "utf-8"
            s = BeautifulSoup(r.text, "html.parser")
            for b in s.select("article.product_pod"):
                rows.append({
                    "title": b.h3.a["title"],
                    "category": cat,
                    "price": float(re.sub(r"[^\d.]", "", b.select_one("p.price_color").text)),
                    "rating": RATING[b.select_one("p.star-rating")["class"][1]],
                })
            nxt = s.select_one("li.next a")
            url = urljoin(url, nxt["href"]) if nxt else None
            time.sleep(0.2)
    return pd.DataFrame(rows)

def synthetic_books(n=1000, seed=42):
    """Stand-in with the same columns, used ONLY if the live site cannot be reached."""
    rng = np.random.default_rng(seed)
    cats = ["Fiction","Mystery","Romance","Science Fiction","Nonfiction","Poetry","History",
            "Fantasy","Childrens","Travel","Music","Business","Science","Humor","Classics"]
    base = dict(zip(cats, rng.uniform(25, 40, len(cats))))
    cat = rng.choice(cats, n)
    rating = rng.integers(1, 6, n)
    words = rng.integers(1, 12, n)
    title = [" ".join("w" * int(l) for l in rng.integers(2, 9, w)) for w in words]
    price = np.clip([base[c] + 0.8*(r-3) + rng.normal(0, 12) for c, r in zip(cat, rating)], 10, 60).round(2)
    return pd.DataFrame({"title": title, "category": cat, "price": price, "rating": rating})

try:
    df = scrape_books(); SOURCE = "live scrape of books.toscrape.com"
except Exception as e:
    print("Scraping failed:", e)
    df = synthetic_books(); SOURCE = "SYNTHETIC fallback (not real data)"
print("Data source:", SOURCE); print(df.shape); df.head()

## 2. Cleaning and Descriptive Statistics

In [ ]:
print(df.info()); print("\nMissing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().dropna().reset_index(drop=True)
df["title_length"] = df["title"].str.len()
df["title_words"] = df["title"].str.split().str.len()
df["high_rating"] = (df["rating"] >= 4).astype(int)
df.describe()

In [ ]:
p = df["price"]
summary = pd.Series({
    "Mean": p.mean(), "Median": p.median(), "Mode": p.mode().iloc[0],
    "Variance": p.var(), "Std Dev": p.std(), "Min": p.min(), "Max": p.max(),
    "Q1": p.quantile(.25), "Q3": p.quantile(.75), "IQR": p.quantile(.75)-p.quantile(.25),
    "Skewness": p.skew(), "Kurtosis": p.kurt()})
print(summary.round(3))
print("\nPrice by rating:"); display(df.groupby("rating")["price"].agg(["count","mean","median","std"]).round(2))
print("Top categories by mean price (min 10 books):")
g = df.groupby("category")["price"].agg(["count","mean"]); display(g[g["count"]>=10].sort_values("mean", ascending=False).head(8).round(2))

## 3. Data Visualisation

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
sns.histplot(df["price"], kde=True, ax=ax[0,0]); ax[0,0].set_title("Price distribution")
sns.boxplot(x="rating", y="price", data=df, ax=ax[0,1]); ax[0,1].set_title("Price by rating")
df["rating"].value_counts().sort_index().plot.bar(ax=ax[0,2], color="steelblue"); ax[0,2].set_title("Books per rating")
df["rating"].value_counts().sort_index().plot.pie(ax=ax[1,0], autopct="%.1f%%"); ax[1,0].set_ylabel(""); ax[1,0].set_title("Rating share")
sns.scatterplot(x="title_length", y="price", hue="rating", data=df, ax=ax[1,1], palette="viridis"); ax[1,1].set_title("Title length vs price")
sns.heatmap(df[["price","rating","title_length","title_words"]].corr(), annot=True, cmap="coolwarm", ax=ax[1,2]); ax[1,2].set_title("Correlation")
plt.tight_layout(); plt.show()

In [ ]:
top = df["category"].value_counts().head(10).index
plt.figure(figsize=(12,5)); sns.boxplot(x="category", y="price", data=df[df.category.isin(top)]); plt.xticks(rotation=40, ha="right")
plt.title("Price by category (10 largest)"); plt.show()
Q1, Q3 = p.quantile([.25,.75]); iqr = Q3-Q1
print("Outliers by IQR rule:", ((p < Q1-1.5*iqr) | (p > Q3+1.5*iqr)).sum())

## 4. Probability and Distributions (Lab 3)

In [ ]:
print("P(rating >= 4) =", round((df.rating>=4).mean(), 4))
print("P(price > 40)  =", round((p>40).mean(), 4))
print("P(price > 40 | rating = 5) =", round((df[df.rating==5].price>40).mean(), 4))
print("P(rating=5 and price>40)  =", round(((df.rating==5)&(p>40)).mean(), 4))
print("Independent? P(A)P(B) =", round((df.rating==5).mean()*(p>40).mean(), 4))
print("\nProbability distribution of rating (X):"); display(df["rating"].value_counts(normalize=True).sort_index().to_frame("P(X=x)").round(4))
pm = df["rating"].value_counts(normalize=True).sort_index(); print("E[X] =", round(float((pm.index.to_numpy()*pm.values).sum()), 3))

In [ ]:
# Binomial: out of 10 random books, how many are highly rated?
q = df["high_rating"].mean(); k = np.arange(0, 11)
plt.bar(k, binom.pmf(k, 10, q)); plt.title(f"Binomial(n=10, p={q:.2f}): highly rated books in 10"); plt.xlabel("k"); plt.show()
print("P(at least 6 of 10 highly rated) =", round(1-binom.cdf(5, 10, q), 4))
# Normal approximation to price
mu, sd = p.mean(), p.std()
print("Normal model: P(20 < price < 40) =", round(norm.cdf(40,mu,sd)-norm.cdf(20,mu,sd), 4), "| empirical =", round(((p>20)&(p<40)).mean(), 4))
print("Shapiro-Wilk normality p-value (sample of 500):", round(stats.shapiro(p.sample(min(500,len(p)), random_state=1)).pvalue, 5))

## 5. Sampling Distribution and Central Limit Theorem (Lab 4)

In [ ]:
rng = np.random.default_rng(42); pop_mean, pop_sd = p.mean(), p.std(ddof=0)
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
for a, n in zip(ax, [5, 30, 100, 300]):
    means = [rng.choice(p, n).mean() for _ in range(2000)]
    a.hist(means, bins=30, color="teal"); a.axvline(pop_mean, color="r"); a.set_title(f"n={n}  SE={np.std(means):.2f}\n(theory {pop_sd/np.sqrt(n):.2f})")
plt.suptitle("Sampling distribution of mean price: spread shrinks as n grows"); plt.tight_layout(); plt.show()
print("Population skewness:", round(p.skew(),3), "-> sample means become approximately normal (CLT).")

## 6. Confidence Intervals and Hypothesis Tests (Lab 5)

In [ ]:
s = df["price"].sample(100, random_state=7); n = len(s); m, sd = s.mean(), s.std()
t_ci = stats.t.interval(0.95, n-1, loc=m, scale=sd/np.sqrt(n))
print(f"Sample mean = {m:.2f}; 95% t-CI = ({t_ci[0]:.2f}, {t_ci[1]:.2f}); true mean = {p.mean():.2f}")
full = stats.t.interval(0.95, len(p)-1, loc=p.mean(), scale=p.sem()); print("95% CI using all books:", tuple(round(float(x),2) for x in full))
# Z-test (sigma known = population sd)
mu0 = 35; z = (m-mu0)/(p.std(ddof=0)/np.sqrt(n)); pz = 2*(1-norm.cdf(abs(z)))
print(f"\nZ-test  H0: mean = {mu0}   z = {z:.3f}, p = {pz:.4f} ->", "Reject H0" if pz<.05 else "Fail to reject H0")
t, pt = stats.ttest_1samp(s, popmean=mu0)
print(f"t-test  H0: mean = {mu0}   t = {t:.3f}, p = {pt:.4f} ->", "Reject H0" if pt<.05 else "Fail to reject H0")
# proportion
k = (s.index.map(df["rating"]) >= 4).sum(); print("\nProportion highly rated in sample:", k/n, "| binomial test p =", round(stats.binomtest(int(k), n, 0.4).pvalue, 4), "(H0: p=0.4)")

## 7. Chi-square, t-test, ANOVA and Correlation (Lab 6)

In [ ]:
alpha = 0.05
def verdict(p): return "Reject H0" if p < alpha else "Fail to reject H0"
# Two-sample t-test: price of high-rated vs low-rated
a, b = df[df.high_rating==1].price, df[df.high_rating==0].price
t, pv = stats.ttest_ind(a, b, equal_var=False); print(f"Welch t-test (rating>=4 vs <4): means {a.mean():.2f} vs {b.mean():.2f}, t={t:.3f}, p={pv:.4f} -> {verdict(pv)}")
# One-way ANOVA: price across ratings
groups = [g.price.values for _, g in df.groupby("rating")]; F, pv = stats.f_oneway(*groups)
print(f"ANOVA price ~ rating: F={F:.3f}, p={pv:.4f} -> {verdict(pv)}")
# ANOVA across categories (those with >=10 books)
big = df.groupby("category").filter(lambda g: len(g) >= 10); F, pv = stats.f_oneway(*[g.price.values for _, g in big.groupby("category")])
print(f"ANOVA price ~ category: F={F:.3f}, p={pv:.4f} -> {verdict(pv)}")
# Chi-square: price band vs rating
df["price_band"] = pd.qcut(df["price"], 3, labels=["Low","Mid","High"])
table = pd.crosstab(df["price_band"], df["rating"]); chi2, pv, dof, exp = chi2_contingency(table)
print(f"Chi-square price_band x rating: chi2={chi2:.3f}, dof={dof}, p={pv:.4f} -> {verdict(pv)}"); display(table)
# Correlation
r, pr = stats.pearsonr(df.price, df.rating); rho, ps = stats.spearmanr(df.price, df.rating)
print(f"Pearson r={r:.3f} (p={pr:.4f}); Spearman rho={rho:.3f} (p={ps:.4f})")

## 8. Regression: Predicting Price (Labs 7 and 8A)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Simple linear regression: price ~ title_length
lr1 = stats.linregress(df.title_length, df.price)
print(f"Simple regression: price = {lr1.intercept:.2f} + {lr1.slope:.4f} * title_length   R2={lr1.rvalue**2:.4f}  p={lr1.pvalue:.4f}")
plt.figure(figsize=(6,4)); sns.regplot(x="title_length", y="price", data=df, scatter_kws={"alpha":.3}); plt.show()

In [ ]:
num, cat = ["rating","title_length","title_words"], ["category"]
X, y = df[num+cat], df["price"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
pre = ColumnTransformer([("n", StandardScaler(), num), ("c", OneHotEncoder(handle_unknown="ignore"), cat)])
models = {"Multiple Linear": LinearRegression(), "Ridge": Ridge(alpha=1.0), "Lasso": Lasso(alpha=0.01, max_iter=10000),
          "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42),
          "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)}
res, fitted = [], {}
for name, m in models.items():
    pipe = Pipeline([("pre", pre), ("m", m)]).fit(Xtr, ytr); pr = pipe.predict(Xte); fitted[name] = pipe
    res.append([name, mean_absolute_error(yte, pr), np.sqrt(mean_squared_error(yte, pr)), r2_score(yte, pr)])
res = pd.DataFrame(res, columns=["Model","MAE","RMSE","R2"]).sort_values("RMSE").round(3); display(res)
best_reg = res.iloc[0]["Model"]; print("Best regression model (lowest RMSE):", best_reg)
reg_baseline = float(np.sqrt(mean_squared_error(yte, np.full(len(yte), ytr.mean()))))
print("Baseline RMSE (always predict mean):", round(reg_baseline, 3))

In [ ]:
# OLS coefficients with significance (multiple regression)
import statsmodels.api as sm
Xo = pd.get_dummies(df[num+cat], columns=cat, drop_first=True).astype(float)
ols = sm.OLS(df["price"], sm.add_constant(Xo, has_constant="add")).fit()
print("R2 =", round(ols.rsquared,4), "| adj R2 =", round(ols.rsquared_adj,4), "| F-test p =", round(ols.f_pvalue,4))
display(pd.DataFrame({"coef": ols.params, "p-value": ols.pvalues}).loc[["const"]+num].round(4))

## 9. Classification: Is a Book Highly Rated? (Lab 8B)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

cnum = ["price","title_length","title_words"]
Xc, yc = df[cnum+cat], df["high_rating"]
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)
cpre = ColumnTransformer([("n", StandardScaler(), cnum), ("c", OneHotEncoder(handle_unknown="ignore"), cat)])
clfs = {"Logistic Regression": LogisticRegression(max_iter=1000), "KNN": KNeighborsClassifier(5), "SVM": SVC(),
        "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)}
out, cfit, cms = [], {}, {}
for name, m in clfs.items():
    pipe = Pipeline([("pre", cpre), ("m", m)]).fit(Xtr, ytr); pr = pipe.predict(Xte); cfit[name] = pipe; cms[name] = confusion_matrix(yte, pr)
    out.append([name, accuracy_score(yte,pr), precision_score(yte,pr,zero_division=0), recall_score(yte,pr), f1_score(yte,pr)])
out = pd.DataFrame(out, columns=["Model","Accuracy","Precision","Recall","F1"]).sort_values("Accuracy", ascending=False).round(3); display(out)
clf_baseline = float(max(yte.mean(), 1-yte.mean()))
print("Majority-class baseline accuracy:", round(clf_baseline, 3))
best = out.iloc[0]["Model"]; pr = cfit[best].predict(Xte)
print("Best classifier:", best); print(classification_report(yte, pr, zero_division=0))
sns.heatmap(confusion_matrix(yte, pr), annot=True, fmt="d", cmap="Blues"); plt.title(f"Confusion matrix - {best}"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

## 10. Export for the Streamlit App and Conclusions
This cell saves two files that `app.py` needs: `books_data.csv` (data) and `book_models.pkl` (all trained models + their evaluation results).

In [ ]:
import joblib, sklearn

df.drop(columns=["price_band"]).to_csv("books_data.csv", index=False)

# Everything app.py needs, in one pickle file
bundle = {
    "sklearn_version": sklearn.__version__,
    "reg_models": fitted,              # {name: fitted regression Pipeline}
    "reg_results": res,                # DataFrame: Model, MAE, RMSE, R2 (sorted by RMSE)
    "reg_baseline": reg_baseline,      # RMSE of always predicting the mean
    "clf_models": cfit,                # {name: fitted classification Pipeline}
    "clf_results": out,                # DataFrame: Model, Accuracy, Precision, Recall, F1
    "clf_confusion": cms,              # {name: confusion matrix (test set)}
    "clf_baseline": clf_baseline,      # majority-class accuracy
}
joblib.dump(bundle, "book_models.pkl", compress=3)

import os
print(f"Saved books_data.csv ({len(df)} rows) and book_models.pkl ({os.path.getsize('book_models.pkl')/1e6:.2f} MB).")
print("Data source:", SOURCE, "| scikit-learn", sklearn.__version__)
try:
    from google.colab import files
    files.download("books_data.csv"); files.download("book_models.pkl")
except ImportError:
    pass


### Conclusions
Read the p-values, R² and accuracy printed above to answer the research questions:
- **Price vs rating:** see the Pearson/Spearman correlation, ANOVA and chi-square results in Section 7.
- **Price across categories:** see the category ANOVA in Section 7.
- **Predictability:** compare each model's R² and RMSE against the mean-prediction baseline (Section 8), and each classifier's accuracy against the majority-class baseline (Section 9). If a model barely beats its baseline, price and rating carry little information about each other.

**Limitations:** the site is a practice catalogue with randomly assigned prices and ratings, so weak relationships are expected. Features such as title length are crude, and results apply only to this catalogue.